In [30]:
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

### Question and Answering

In [31]:
content = """
    The solar system consists of the Sun, eight planets, their moons, dwarf planets, and smaller objects like asteroids and comets. 
    The inner planets—Mercury, Venus, Earth, and Mars—are rocky and solid. 
    The outer planets—Jupiter, Saturn, Uranus, and Neptune—are much larger and gaseous.
"""


In [39]:
question = "What are the inner planets in the solar system?"

template = """
Answer the {question} based on the following context:{content}.
Respond "Unsure about answers", if not sure about the answer.

Answer:


"""



In [40]:
prompt = PromptTemplate.from_template(template)

In [41]:
def format_prompt(variables):
    return prompt.format(**variables)

In [42]:
format_prompt({"question": question, "content": content})

'\nAnswer the What are the inner planets in the solar system? based on the following context:\n    The solar system consists of the Sun, eight planets, their moons, dwarf planets, and smaller objects like asteroids and comets. \n    The inner planets—Mercury, Venus, Earth, and Mars—are rocky and solid. \n    The outer planets—Jupiter, Saturn, Uranus, and Neptune—are much larger and gaseous.\n.\nRespond "Unsure about answers", if not sure about the answer.\n\nAnswer:\n\n\n'

In [43]:
llm = ChatOllama(
    model="qwen3:latest",
    base_url = "http://10.88.150.240:11434",
    temperature=0.3,
)

In [44]:
qa_chain =(RunnableLambda(format_prompt)
            |llm
            | StrOutputParser()
)

In [45]:
qa_chain.invoke({"question": question, "content": content})

'The inner planets in the solar system are Mercury, Venus, Earth, and Mars. These planets are rocky and solid, as stated in the context. \n\nAnswer: Mercury, Venus, Earth, and Mars.'

### Test classification
Here is a text classifcation agent designed to categorze text into predefined categories

In [47]:
text = """
the concert last night was an exhilarating experience with outstand performance by all artists.

"""
categories = "Entertainment, Food and Dining, Technology, Literature, Music"

template = """ 
Classify the {text} into one of the {categoris}.

Category:

"""

prompt = PromptTemplate.from_template(template)

classification_chain = RunnableLambda(format_prompt) | llm | StrOutputParser()

classification_chain.invoke({"text": text, "categoris": categories})



'Category: **Music**  \n\nThe sentence describes a concert, which is a live music performance, and highlights the artists\' outstanding performance, directly aligning with the **Music** category. While concerts can fall under "Entertainment" broadly, the specific focus on musical performance and artists makes **Music** the most precise classification.'

In [49]:
description = """
Retrieve the names and email addresses of all customers from the 'customers' table who have made a purchase in the last 30 days.
The table 'purchases' contains a column 'purchase_date'

"""

template = """
Genearate an SQL query based on description: {description}

SQL Query:

"""

prompt = PromptTemplate.from_template(template)

sql_chain = RunnableLambda(format_prompt) | llm | StrOutputParser()
sql_chain.invoke({"description": description})


"To retrieve the names and email addresses of all customers who have made a purchase in the last 30 days, we can use a combination of a `JOIN` or an `EXISTS` subquery. The `EXISTS` approach is often more efficient and avoids duplicates, which aligns with the requirement to return each customer only once, regardless of how many purchases they made in the specified time frame.\n\nHere is the SQL query that accomplishes this:\n\n```sql\nSELECT c.name, c.email\nFROM customers c\nWHERE EXISTS (\n    SELECT 1\n    FROM purchases p\n    WHERE p.customer_id = c.customer_id\n    AND p.purchase_date >= CURRENT_DATE - INTERVAL '30 days'\n);\n```\n\n---\n\n### Explanation:\n\n- **`SELECT c.name, c.email`**: Retrieves the customer names and email addresses from the `customers` table.\n- **`FROM customers c`**: Specifies the `customers` table as the source of customer data.\n- **`WHERE EXISTS (...)`**: Ensures that only customers who have at least one purchase in the last 30 days are included.\n  - 

### Role playing

In [50]:
role = """
    Dungeon & Dragons game master
"""

tone = "engaging and immersive"

template = """
    You are an expert {role}. I have this question {question}. I would like our conversation to be {tone}.
    
    Answer:
    
"""
prompt = PromptTemplate.from_template(template)

# Create the LCEL chain
roleplay_chain = (
    RunnableLambda(format_prompt)
    | llm 
    | StrOutputParser()
)

# Create an interactive chat loop
while True:
    query = input("Question: ")
    
    if query.lower() in ["quit", "exit", "bye"]:
        print("Answer: Goodbye!")
        break
        
    response = roleplay_chain.invoke({"role": role, "question": query, "tone": tone})
    print("Answer: ", response)

Answer:  Ah, a fellow adventurer! 🎲✨ I'm thrilled to help you craft an unforgettable D&D journey. Whether you're preparing for your first campaign, troubleshooting a tricky scenario, or seeking inspiration for your next epic tale, I'm here to lend my wisdom and enthusiasm.  

So—what's on your mind? Are you:  
- **Prepping a new campaign** and need help designing a world or plot?  
- **Running a session** and encountering a challenge you'd like to troubleshoot?  
- **Creating characters** and want guidance on making them unique and compelling?  
- **Exploring lore** or rules and need clarity on the *Player's Handbook* or *Dungeon Master's Guide*?  

Tell me your question, and let’s dive into the magic together! 🌟  
*(And if you’re unsure where to start? I’ll help you chart your course!)*
Answer: Goodbye!
